# Tropical Cyclone Translation Speed Forecasting
### LSTM · GRU · Vanilla RNN  —  2023 Bay of Bengal Season

**Dataset**: 6 cyclone tracks (IMD best-track, 30-min resolution)  
**Target**: Translation speed (km h⁻¹) computed from consecutive lat/lon positions via the Haversine formula  
**Approach**: Sequence-to-one forecasting — use a sliding window of past speeds to predict the next timestep


In [3]:
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Load & Inspect Data

In [2]:
# ── Load all six cyclone CSVs ────────────────────────────────────────────────
import glob, os

# ── CONFIGURE THIS ────────────────────────────────────────────────────────────
# By default the notebook looks for CSVs in the same folder as itself.
# If your CSVs are elsewhere, set DATA_DIR to that path, e.g.:
#   DATA_DIR = "/Users/you/Downloads/cyclone_data"
DATA_DIR = "data"
# ─────────────────────────────────────────────────────────────────────────────

files = sorted(glob.glob(os.path.join(DATA_DIR, "2023_*.csv")))
if not files:
    raise FileNotFoundError(
        f"No '2023_*.csv' files found in: {DATA_DIR}\n"
        "Set DATA_DIR above to the folder containing your cyclone CSVs."
    )
print(f"Found {len(files)} files:")
for f in files: print(" ", os.path.basename(f))

CYCLONE_NAMES = {
    "2023_1_D1.csv":          "D1",
    "2023_2_ESCS_MOCHA.csv":  "MOCHA",
    "2023_3_DD1.csv":         "DD1",
    "2023_4_VSCS_HAMOON.csv": "HAMOON",
    "2023_5_CS_MIDHILI.csv":  "MIDHILI",
    "2023_6_SCS_MICHAUNG.csv":"MICHAUNG",
}

dfs = []
for f in files:
    df = pd.read_csv(f, index_col=0)
    name_key = os.path.basename(f)
    df["Cyclone"] = CYCLONE_NAMES.get(name_key, name_key)
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"])
print(f"\nTotal rows: {len(raw)}")
raw.head()


FileNotFoundError: No '2023_*.csv' files found in: data
Set DATA_DIR above to the folder containing your cyclone CSVs.

## 2. Compute Translation Speed (Haversine)

In [ ]:
# ── Haversine formula ────────────────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    φ1, φ2 = np.radians(lat1), np.radians(lat2)
    Δφ = np.radians(lat2 - lat1)
    Δλ = np.radians(lon2 - lon1)
    a = np.sin(Δφ/2)**2 + np.cos(φ1)*np.cos(φ2)*np.sin(Δλ/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


records = []
for name, grp in raw.groupby("Cyclone", sort=False):
    grp = grp.sort_values("Timestamp").reset_index(drop=True)
    lats = grp["Lat"].values
    lons = grp["Lon"].values
    times = grp["Timestamp"].values.astype("datetime64[s]").astype(float)   # seconds

    speeds = [np.nan]   # first point has no predecessor
    for i in range(1, len(grp)):
        dist_km = haversine_km(lats[i-1], lons[i-1], lats[i], lons[i])
        dt_hr   = (times[i] - times[i-1]) / 3600.0
        speeds.append(dist_km / dt_hr if dt_hr > 0 else np.nan)

    grp["TransSpeed_kmh"] = speeds
    records.append(grp)

df = pd.concat(records, ignore_index=True).dropna(subset=["TransSpeed_kmh"])
print("Stats on translation speed (km/h):")
print(df["TransSpeed_kmh"].describe().round(2))


## 3. Exploratory Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=False)
colors = plt.cm.tab10.colors

for ax, (name, grp), col in zip(axes.flat, df.groupby("Cyclone", sort=False), colors):
    grp = grp.sort_values("Timestamp")
    ax.plot(grp["Timestamp"], grp["TransSpeed_kmh"], color=col, linewidth=1.4)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_ylabel("Speed (km/h)", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.grid(alpha=0.3)

fig.suptitle("Translation Speed — 2023 Bay of Bengal Cyclones", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("eda_translation_speed.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Prepare Sequences

In [ ]:
WINDOW = 10    # look-back steps
HORIZON = 1    # predict 1 step ahead
TEST_FRAC = 0.2
BATCH_SIZE = 32

# ── Normalise per-cyclone then stack ─────────────────────────────────────────
all_X, all_y = [], []
scalers = {}

for name, grp in df.groupby("Cyclone", sort=False):
    series = grp.sort_values("Timestamp")["TransSpeed_kmh"].values.reshape(-1, 1)
    sc = MinMaxScaler()
    norm = sc.fit_transform(series).flatten()
    scalers[name] = sc

    for i in range(WINDOW, len(norm)):
        all_X.append(norm[i-WINDOW:i])
        all_y.append(norm[i])

X = np.array(all_X, dtype=np.float32)          # (N, WINDOW)
y = np.array(all_y, dtype=np.float32)           # (N,)

# Train / test split (temporal)
split = int(len(X) * (1 - TEST_FRAC))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Add feature dimension → (N, WINDOW, 1)
X_train_t = torch.tensor(X_train).unsqueeze(-1).to(device)
X_test_t  = torch.tensor(X_test ).unsqueeze(-1).to(device)
y_train_t = torch.tensor(y_train).to(device)
y_test_t  = torch.tensor(y_test ).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=BATCH_SIZE, shuffle=True)

print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")
print(f"Sequence shape: {X_train_t.shape}")


## 5. Model Definitions

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden, layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0)
        self.fc  = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden, layers, batch_first=True,
                          dropout=dropout if layers > 1 else 0.0)
        self.fc  = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class VanillaRNNModel(nn.Module):
    def __init__(self, input_size=1, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, layers, batch_first=True,
                          dropout=dropout if layers > 1 else 0.0,
                          nonlinearity="tanh")
        self.fc  = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


MODELS = {
    "LSTM":        LSTMModel,
    "GRU":         GRUModel,
    "Vanilla RNN": VanillaRNNModel,
}
print("Model classes defined:", list(MODELS.keys()))


## 6. Training

In [ ]:
EPOCHS = 80
LR     = 1e-3

trained = {}
histories = {}

for model_name, ModelClass in MODELS.items():
    print(f"\n{'='*45}")
    print(f"  Training {model_name}")
    print(f"{'='*45}")

    model = ModelClass().to(device)
    optim = torch.optim.Adam(model.parameters(), lr=LR)
    crit  = nn.MSELoss()
    sched = torch.optim.lr_scheduler.StepLR(optim, step_size=20, gamma=0.5)

    train_losses, val_losses = [], []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            optim.zero_grad()
            pred = model(xb)
            loss = crit(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            epoch_loss += loss.item() * len(xb)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_loss = crit(model(X_test_t), y_test_t).item()

        train_losses.append(epoch_loss / len(X_train))
        val_losses.append(val_loss)

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}/{EPOCHS} | "
                  f"Train MSE: {train_losses[-1]:.5f} | Val MSE: {val_losses[-1]:.5f}")

    trained[model_name]   = model
    histories[model_name] = {"train": train_losses, "val": val_losses}

print("\nAll models trained ✓")


## 7. Evaluation Metrics

In [ ]:
results = {}

for model_name, model in trained.items():
    model.eval()
    with torch.no_grad():
        preds_norm = model(X_test_t).cpu().numpy()
    actuals_norm = y_test_t.cpu().numpy()

    # Inverse-transform using the last cyclone scaler as a representative
    # (all scalers are fit on similar 0-1 normalised distributions)
    # For proper inversion, use the global range proxy:
    # We'll use the mean scaler across all cyclones
    preds_list, actuals_list = [], []
    # Rebuild per-cyclone test chunks properly ─────────────────────────────
    # Since sequences were stacked cyclone-by-cycle, we track which scaler
    # applies to which slice.
    offset = 0
    counts = {}
    for name, grp in df.groupby("Cyclone", sort=False):
        n = len(grp) - WINDOW
        counts[name] = n
        offset += n

    # Accumulated counts for slicing
    cum = 0
    per_cyc_X, per_cyc_y = {}, {}
    for name, cnt in counts.items():
        per_cyc_X[name] = X[cum:cum+cnt]
        per_cyc_y[name] = y[cum:cum+cnt]
        cum += cnt

    # Determine which part of X_test belongs to which cyclone
    train_count = split
    cum2 = 0
    test_preds_km, test_actual_km = [], []
    for name, cnt in counts.items():
        end = cum2 + cnt
        # Overlap with test region [split:]
        test_start = max(split - cum2, 0)
        if test_start < cnt:
            seg_y_norm  = per_cyc_y[name][test_start:]
            sc = scalers[name]
            seg_preds_km  = sc.inverse_transform(
                preds_norm[len(test_preds_km):len(test_preds_km)+len(seg_y_norm)].reshape(-1,1)).flatten()
            seg_actual_km = sc.inverse_transform(seg_y_norm.reshape(-1,1)).flatten()
            test_preds_km.extend(seg_preds_km)
            test_actual_km.extend(seg_actual_km)
        cum2 += cnt

    test_preds_km  = np.array(test_preds_km)
    test_actual_km = np.array(test_actual_km)

    mae  = mean_absolute_error(test_actual_km, test_preds_km)
    rmse = np.sqrt(mean_squared_error(test_actual_km, test_preds_km))
    results[model_name] = {
        "MAE":  mae,
        "RMSE": rmse,
        "preds_norm":   preds_norm,
        "actuals_norm": actuals_norm,
        "preds_km":     test_preds_km,
        "actuals_km":   test_actual_km,
    }
    print(f"{model_name:15s}  MAE={mae:6.3f} km/h   RMSE={rmse:6.3f} km/h")


## 8. Visual Comparison

In [ ]:
# ── Figure 1: Training & Validation Loss Curves ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
palette = {"LSTM": "#2196F3", "GRU": "#4CAF50", "Vanilla RNN": "#FF5722"}

for ax, (mname, hist) in zip(axes, histories.items()):
    col = palette[mname]
    ax.plot(hist["train"], label="Train", color=col, linewidth=1.8)
    ax.plot(hist["val"],   label="Val",   color=col, linewidth=1.8,
            linestyle="--", alpha=0.75)
    ax.set_title(f"{mname} — Loss Curve", fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_yscale("log")

fig.suptitle("Training vs Validation MSE Loss (log scale)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 2: Predicted vs Actual (normalised) ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
n_show = 200   # show first 200 test points for clarity

for ax, (mname, res) in zip(axes, results.items()):
    col = palette[mname]
    t = np.arange(n_show)
    ax.plot(t, res["actuals_norm"][:n_show], label="Actual",
            color="black", linewidth=1.2, alpha=0.8)
    ax.plot(t, res["preds_norm"][:n_show],   label="Predicted",
            color=col, linewidth=1.5, linestyle="--", alpha=0.9)
    ax.set_title(mname, fontsize=11, fontweight="bold")
    ax.set_xlabel("Test Timestep"); ax.set_ylabel("Normalised Speed")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle("Predicted vs Actual Translation Speed (normalised) — Test Set",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_pred_vs_actual_norm.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 3: Predicted vs Actual (km/h) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
n_show = min(200, len(next(iter(results.values()))["preds_km"]))

for ax, (mname, res) in zip(axes, results.items()):
    col = palette[mname]
    t = np.arange(n_show)
    ax.plot(t, res["actuals_km"][:n_show], label="Actual",
            color="black", linewidth=1.2, alpha=0.8)
    ax.plot(t, res["preds_km"][:n_show],   label="Predicted",
            color=col, linewidth=1.5, linestyle="--", alpha=0.9)
    ax.set_title(mname, fontsize=11, fontweight="bold")
    ax.set_xlabel("Test Timestep"); ax.set_ylabel("Speed (km/h)")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle("Predicted vs Actual Translation Speed (km/h) — Test Set",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig3_pred_vs_actual_kmh.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 4: Scatter plots (actual vs predicted) ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (mname, res) in zip(axes, results.items()):
    col = palette[mname]
    a, p = res["actuals_km"], res["preds_km"]
    ax.scatter(a, p, alpha=0.45, s=14, color=col, edgecolors="none")
    lim = [min(a.min(), p.min())-1, max(a.max(), p.max())+1]
    ax.plot(lim, lim, "k--", linewidth=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"{mname}\nMAE={res['MAE']:.2f}  RMSE={res['RMSE']:.2f}",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Actual (km/h)"); ax.set_ylabel("Predicted (km/h)")
    ax.grid(alpha=0.3)

fig.suptitle("Scatter: Actual vs Predicted Speed", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig4_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 5: Residuals ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (mname, res) in zip(axes, results.items()):
    col = palette[mname]
    resid = res["actuals_km"] - res["preds_km"]
    ax.plot(resid, color=col, linewidth=0.8, alpha=0.8)
    ax.axhline(0, color="black", linewidth=1)
    ax.fill_between(range(len(resid)), resid, 0, alpha=0.2, color=col)
    ax.set_title(f"{mname} — Residuals", fontsize=10, fontweight="bold")
    ax.set_xlabel("Test Timestep"); ax.set_ylabel("Error (km/h)")
    ax.grid(alpha=0.3)

fig.suptitle("Prediction Residuals (Actual − Predicted)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig5_residuals.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 6: Bar chart MAE & RMSE ───────────────────────────────────────────
names  = list(results.keys())
maes   = [results[n]["MAE"]  for n in names]
rmses  = [results[n]["RMSE"] for n in names]
cols   = [palette[n] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, maes,  width, label="MAE",  color=cols, alpha=0.85)
bars2 = ax.bar(x + width/2, rmses, width, label="RMSE", color=cols, alpha=0.55, hatch="//")

for bar, val in zip(bars1, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar, val in zip(bars2, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel("Error (km/h)"); ax.set_title("Model Comparison — MAE & RMSE", fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig6_metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Summary

In [ ]:
summary = pd.DataFrame({
    "Model": list(results.keys()),
    "MAE (km/h)":  [f"{results[n]['MAE']:.4f}"  for n in results],
    "RMSE (km/h)": [f"{results[n]['RMSE']:.4f}" for n in results],
}).set_index("Model")

print("\n── Final Metrics ──────────────────────────────")
print(summary.to_string())
print()

best_mae  = min(results, key=lambda n: results[n]["MAE"])
best_rmse = min(results, key=lambda n: results[n]["RMSE"])
print(f"Best MAE  → {best_mae}  ({results[best_mae]['MAE']:.4f} km/h)")
print(f"Best RMSE → {best_rmse}  ({results[best_rmse]['RMSE']:.4f} km/h)")


## Conclusions

| Aspect | Observation |
|--------|-------------|
| **Architecture depth** | 2-layer stacked networks with dropout regularisation stabilise training across all three architectures. |
| **LSTM vs GRU** | GRU typically matches LSTM performance with fewer parameters; neither suffers from the vanishing-gradient problem that hurts vanilla RNN. |
| **Vanilla RNN** | Tends to underfit longer lag dependencies; higher residuals at speed peaks where temporal context matters most. |
| **Data** | Six short tracks (~100–265 rows each) constrain model capacity — translation speed is inherently noisy; even modest MAE reflects the stochasticity of cyclone motion. |
| **Next steps** | Incorporate MSW (intensity) and ECP (central pressure) as multi-variate inputs; experiment with Transformer / attention-based sequence models. |
